# Training model without any preprocessing or feature engineering

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

## Load Dataset

In [22]:
df = pd.read_csv('../quora_question_pair.csv')
print(df.shape)
df.head()

(404290, 6)


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [23]:
df = df.sample(100000)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 155926 to 135813
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            100000 non-null  int64 
 1   qid1          100000 non-null  int64 
 2   qid2          100000 non-null  int64 
 3   question1     100000 non-null  object
 4   question2     100000 non-null  object
 5   is_duplicate  100000 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 5.3+ MB


In [25]:
df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [26]:
df.dropna(inplace=True)

In [27]:
df.duplicated().sum()

np.int64(0)

In [28]:
ques_df = df[['question1','question2']]
ques_df.head()

,question1,question2
155926,What is a suitable solar panel installation pr...,Which is a good solar panel installation provi...
303997,What bachelor degree should?,What do I do with a bachelors degree?
331975,Who was a better Dumbledore -- Richard Harris ...,"Which portrayal Dumbledore did you prefer, Ric..."
136330,How did Tom Hardy build his body for the role ...,"In The Dark Knight Rises, why did Batman lose ..."
87627,You have a group in which an individual can't ...,My friend is getting bullied by a girl that bu...


In [29]:
questions = list(ques_df['question1']) + list(ques_df['question2'])
cv = CountVectorizer(max_features = 3000)

In [30]:
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(),2)

In [31]:
temp_df1 = pd.DataFrame(q1_arr, index = ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index = ques_df.index)
temp_df = pd.concat([temp_df1,temp_df2],axis=1)
temp_df.shape

(100000, 6000)

In [32]:
temp_df['is_duplicate'] = df['is_duplicate']

In [33]:
temp_df.shape

(100000, 6001)

In [36]:
x = temp_df.iloc[:,:-1]
y = temp_df.iloc[:,-1]
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=1)

In [ ]:
x_train.columns = [f"feature_{i}" for i in range(x_train.shape[1])]
x_test.columns = [f"feature_{i}" for i in range(x_test.shape[1])]

In [37]:
rf = RandomForestClassifier()
rf.fit(x_train,y_train)
y_pred = rf.predict(x_test)
accuracy_score(y_test, y_pred)

0.76475

In [ ]:
xgb = XGBClassifier()
xgb.fit(x_train, y_train)
y_pred = xgb.predict(x_test)
accuracy_score(y_test, y_pred)

0.73325